In [ ]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
%pip install kaggle
%pip install kagglehub[pandas-datasets]
%pip install --upgrade --force-reinstall --no-deps kaggle

StatementMeta(, 92f83521-3160-4a5f-b897-9b616a60e362, 10, Finished, Available, Finished, False)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8/8 [kaggle]2m6/8 [jupytext]]

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /nfs4/pyenv-c09317cc-7c95-4755-9262-49ab99898250/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /nfs4/pyenv-c09317cc-7c95-4755-9262-49ab99898250/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Using cached kaggle-2.2.4-py3-none-any.whl.metadata (17 kB)
Using cached kaggle-2.2.4-py3-none-any.whl (126 kB)
  Attempting uninstall: kaggle
    Found existing installation: kaggle 2.2.4
    Uninstalling kaggle-2.2.4:
      Successfully uninstalled kaggle-2.2.4

[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: /nfs4/pyenv-c09317cc-7c95-4755-9262-49ab99898250/bin/python -m pip install --upgrade pip


**Protegemos las credenciales de la API de Keaggle usando la biblioteca de variables de MS Fabric para no usar servicios de Azure Key Vaul**t

In [ ]:
# Cargar la biblioteca de variables 
vl = notebookutils.variableLibrary.getLibrary("Variables_1")
USER = vl.getVariable("USER")
PSSWD = vl.getVariable("PASSWD")


StatementMeta(, 92f83521-3160-4a5f-b897-9b616a60e362, 12, Finished, Available, Finished, False)

In [ ]:
import os,shutil, re

os.environ["KAGGLE_USERNAME"] = USER
os.environ["KAGGLE_KEY"] = PSSWD

StatementMeta(, 92f83521-3160-4a5f-b897-9b616a60e362, 13, Finished, Available, Finished, False)

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter
from kaggle.api.kaggle_api_extended import KaggleApi


StatementMeta(, 92f83521-3160-4a5f-b897-9b616a60e362, 14, Finished, Available, Finished, False)

**Se copian los datos en csv y parquet en modo overwrite, luego se transforman a delta en overwrite por lo cual NO se asume carga incremental ni watermarks ya que el origen se asume como data estatica.**

In [ ]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# 2. Descargar el dataset completo
local_path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")
print("Descargado en:", local_path)

# 3. Copiar los CSV al Lakehouse (Files/olist/csv)
csv_dest = "/lakehouse/default/Files/olist/csv"
os.makedirs(csv_dest, exist_ok=True)

csv_files = [f for f in os.listdir(local_path) if f.endswith(".csv")]
for f in csv_files:
    shutil.copy(os.path.join(local_path, f), os.path.join(csv_dest, f))

# 4. Convertir cada CSV a parquet (Files/olist/parquet)
base_abfss = "abfss://PRUEBA_NS@onelake.dfs.fabric.microsoft.com/RAW.Lakehouse/Files/olist"

for f in csv_files:
    name = re.sub(r"\.csv$", "", f)   # ej: olist_orders_dataset
    df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .option("multiLine", True)
        .option("escape", '"')
        .csv(f"Files/olist/csv/{f}")
    )
    (
        df.write
        .mode("overwrite")
        .parquet(f"{base_abfss}/parquet/{name}")
    )
   


    print(f"✔ {name}: {df.count()} filas")

    # Escribir como tabla Delta en el Lakehouse
    df.write.format("delta").mode("overwrite").saveAsTable(name)

StatementMeta(, 92f83521-3160-4a5f-b897-9b616a60e362, 16, Finished, Available, Finished, False)

Descargado en: /home/trusted-service-user/.cache/kagglehub/datasets/olistbr/brazilian-ecommerce/versions/2
✔ olist_customers_dataset: 99441 filas
✔ olist_order_payments_dataset: 103886 filas
✔ olist_products_dataset: 32951 filas
✔ olist_sellers_dataset: 3095 filas
✔ olist_orders_dataset: 99441 filas
✔ olist_order_items_dataset: 112650 filas
✔ product_category_name_translation: 71 filas
✔ olist_order_reviews_dataset: 99224 filas
✔ olist_geolocation_dataset: 1000163 filas
